# SNAIL Measurements — thin driver notebook

All logic lives in the `qlab` package next to this notebook. Each cell below is one measurement = one call.
Parameters shown are the last-used values from the original notebooks (`QLab_tra_HH.ipynb`, `HH_qubit.ipynb`).

**Modes**
- `MODE = 'fake'` — synthetic instruments, runs on any laptop. Resonator at 8.0122 GHz, fake qubit at 5.6 GHz.
- `MODE = 'real'` — lab PC only (needs PycQED + hardware). Same calls, real instruments.

See `README.md` for the old-cell → new-function mapping.

In [ ]:
import qlab

MODE = 'fake'          # 'fake' (laptop) or 'real' (lab PC)

st = qlab.connect_fake() if MODE == 'fake' else qlab.connect()

## 1. Resonator spectroscopy (VNA only)

In [ ]:
# Low-power S21 scan — was QLab_tra cell 10
qlab.resonator_scan(st, center=8.01225e9, span=20e6,
                    power=-30, if_bandwidth=2000, npts=2001,
                    averages=200, delay_t=6.5e-8, measure='S21')

In [ ]:
# S11 reflection — was QLab_tra cell 17 (note the reflection-path delay)
qlab.resonator_scan(st, center=8.22025e9, span=50e6,
                    power=-30, if_bandwidth=1000, npts=2001,
                    averages=100, delay_t=qlab.config.DEFAULT_DELAY_S11,
                    measure='S11')

In [ ]:
# Punch-out: power sweep, dressed -> bare — was QLab_tra cell 12
qlab.resonator_power_sweep(st, center=8.01225e9, span=20e6,
                           power_start=-30, power_stop=20, power_step=1,
                           if_bandwidth=1000, npts=1001, averages=300,
                           delay_t=6.5e-8)

In [ ]:
# Long-term stability: one scan every 2 h — was QLab_tra cell 14
# n_runs=None runs forever; set a number for a bounded test
qlab.stability_monitor(st, center=8.25091e9, span=50e6,
                       interval_s=7200, n_runs=3,
                       power=-30, if_bandwidth=1000, npts=2001,
                       averages=100, delay_t=6.24e-8)

## 2. Two-tone spectroscopy (prebuilt QLab_Neon methods — real hardware only)

In [ ]:
# Pump sweep at fixed probe — was HH_qubit cell 9
qlab.two_tone_pump(st, probe_freq=8.25087e9,
                   pump_freq_start=4.99875e9, pump_freq_stop=4.99875e9 + 2.5e5,
                   pump_f_npts=100, pump_power=-10, probe_power=-25,
                   if_bandwidth=1000, averages=500, delay_t=6.39e-8)

In [ ]:
# 2D: pump frequency x probe frequency — was HH_qubit cell 10
center, span = 8.25087e9, 5e6
qlab.two_tone_probe_sweep(st,
                          pump_freq_start=5e9, pump_freq_stop=6e9, pump_f_npts=1e5,
                          probe_freq_start=center - span/2, probe_freq_stop=center + span/2,
                          probe_f_npts=1001,
                          pump_power=-30, probe_power=-30,
                          if_bandwidth=500, averages=100, delay_t=6.39e-8)

## 3. Fast pump sweep (Gen3, ~2 s/pt) — the workhorse

VNA is configured once as a single-point trace at the probe frequency; the loop only moves the pump and reads I/Q directly. Returns a DataFrame and saves a CSV. `settle_s > 0` adds the settle wait the original notebook flagged as missing.

In [ ]:
# Qubit search — was HH_qubit cell 16
df = qlab.pump_sweep_fast(st, probe_freq=8.0121625e9,
                          pump_freq_start=4.5e9, pump_freq_stop=6.5e9,
                          pump_freq_step=200e3,
                          pump_power=5, vna_power=-20,
                          if_bandwidth=500, averages=300,
                          delay_t=6.62e-8, settle_s=0.0)

## 4. SNAIL tests

In [ ]:
# SNAIL pump sweep (high band) — was HH_qubit cell 30
df = qlab.pump_sweep_fast(st, probe_freq=7.642922e9,
                          pump_freq_start=15.5e9, pump_freq_stop=16.5e9,
                          pump_freq_step=1e5,
                          pump_power=25, vna_power=0,
                          if_bandwidth=500, averages=1000, delay_t=6.4e-8)

In [ ]:
# Kerr shift vs pump power — was HH_qubit cell 32
qlab.kerr_shift_sweep(st, probe_freq=7.642922e9, span=10e6, delta_freq=200e6,
                      pump_power_start=-20, pump_power_stop=0, pump_power_step=1,
                      vna_power=0, if_bandwidth=1000, npts=1001,
                      averages=200, delay_t=6.9e-8)

In [ ]:
# Observation-only helpers — were HH_qubit cells 28 / 33
# qlab.pump_observe(st, resonance_freq=7.643e9, pump_power=0, dwell_s=25)
# qlab.pump_power_ramp(st, pump_freq=7.642922e9 - 500e6,
#                      pump_power_start=-20, pump_power_stop=25, dwell_s=10)

## 5. Shutdown

In [ ]:
qlab.all_off(st)   # pump off, VNA RF off